# 00_setup — Environment Configuration
## ClinVar Medallion Pipeline

Defines all configuration for the pipeline and validates the environment.
Run this first — every other notebook re-runs it via `%run`.

**Sets up:**
- Catalog, schema, and table names (Unity Catalog `catalog.schema.table` convention)
- The volume for the raw download, and the file path
- Pipeline constants (target genome build, significance and pathogenic category lists)
- Delta/Spark settings, with graceful fallback where Serverless locks them
- An environment validation gate (volume, Spark SQL, schema reachable)

In [0]:
# Spark version check. Prints the PySpark version to confirm the engine is on. 
# Databricks Serverless Compute (PySpark 4.x), sparkContext is not available on serverless — as expected.

import pyspark
print(f"PySpark version  : {pyspark.__version__}")
print(f"Spark version    : {spark.version}")
print(f"Compute type     : Serverless (no sparkContext JVM access — expected)")
print(f"Delta Lake       : available via spark.sql and DataFrame API")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# UNITY CATALOG NAMESPACE
# ─────────────────────────────────────────────────────────────────────────────
# CE forces one shared schema; in a full UC workspace bronze/silver/gold
# would be separate schemas with access control. Layer is encoded in the
# table name instead. Tables live under workspace.clinvar.*

CATALOG       = "workspace"
SCHEMA_BRONZE = "clinvar"
SCHEMA_SILVER = "clinvar"
SCHEMA_GOLD   = "clinvar"

# Full table identifiers (used in spark.read / spark.write throughout pipeline)
TABLE_BRONZE = f"{CATALOG}.{SCHEMA_BRONZE}.variant_summary_raw"
TABLE_SILVER = f"{CATALOG}.{SCHEMA_SILVER}.variant_summary_clean"

TABLE_GOLD_SIGNIFICANCE  = f"{CATALOG}.{SCHEMA_GOLD}.significance_summary"
TABLE_GOLD_PATHOGENIC    = f"{CATALOG}.{SCHEMA_GOLD}.pathogenic_by_gene"
TABLE_GOLD_CHROMOSOME    = f"{CATALOG}.{SCHEMA_GOLD}.variants_by_chromosome"
TABLE_GOLD_CONDITION     = f"{CATALOG}.{SCHEMA_GOLD}.condition_burden"

print("Table names configured:")
for name, val in {
    "Bronze":             TABLE_BRONZE,
    "Silver":             TABLE_SILVER,
    "Gold - Significance": TABLE_GOLD_SIGNIFICANCE,
    "Gold - Pathogenic":  TABLE_GOLD_PATHOGENIC,
    "Gold - Chromosome":  TABLE_GOLD_CHROMOSOME,
    "Gold - Condition":   TABLE_GOLD_CONDITION,
}.items():
    print(f"  {name:25s}: {val}")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CREATE UNITY CATALOG SCHEMA AND VOLUME
# ─────────────────────────────────────────────────────────────────────────────
# Must run before any file paths or Delta tables are created.
# Safe to re-run — nothing breaks if the schema or volume already exists.

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.clinvar")
print("  Schema  : workspace.clinvar — ready")

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.clinvar.raw_files")
print("  Volume  : workspace.clinvar.raw_files — ready")

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.clinvar.clean_tables")
print("  Volume  : workspace.clinvar.clean_tables — ready")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# FILE PATHS
# ─────────────────────────────────────────────────────────────────────────────
# Only Bronze needs a file path (the raw download).
# Silver/Gold write Delta tables — Delta manages its own storage.
# Volume path follows UC structure: /Volumes/<catalog>/<schema>/<volume>

VOLUME_ROOT   = f"/Volumes/{CATALOG}/{SCHEMA_BRONZE}/raw_files"
RAW_FILE_NAME = "variant_summary.txt.gz"
RAW_FILE_PATH = f"{VOLUME_ROOT}/{RAW_FILE_NAME}"

print("File paths configured:")
print(f"  Volume root : {VOLUME_ROOT}")
print(f"  Raw file    : {RAW_FILE_PATH}")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# DELTA LAKE CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
# Try to set Delta/Spark options; Serverless locks some, so each is wrapped
# in try/except — locked settings print "managed by platform" instead of failing.
# timeParserPolicy is set to LEGACY here; Silver still parses LastEvaluated via a
# Python UDF (strptime), which is immune to this Spark setting either way.

try:
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
    print("  autoMerge       : set explicitly")
except Exception as e:
    print(f"  autoMerge       : managed by platform ({type(e).__name__})")

try:
    spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
    print("  optimizeWrite   : set explicitly")
except Exception as e:
    print(f"  optimizeWrite   : managed by platform ({type(e).__name__})")

print("\nDelta Lake is available — configs either set or platform-managed.")

try:
    spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
    print("  timeParserPolicy: set to LEGACY")
except Exception as e:
    print(f"  timeParserPolicy: managed by platform ({type(e).__name__})")

# ─────────────────────────────────────────────────────────────────────────────
# PIPELINE CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
# GENOME_ASSEMBLY: Silver filters to this build to drop duplicate variants.
# SIGNIFICANCE_CATEGORIES: canonical clinical categories used downstream.

GENOME_ASSEMBLY = "GRCh38"

SIGNIFICANCE_CATEGORIES = [
    "Pathogenic",
    "Likely pathogenic",
    "Uncertain significance",
    "Likely benign",
    "Benign",
    "Conflicting interpretations of pathogenicity",
    "Other / Not provided",
]

# Clinical categories treated as "actionable" pathogenic burden across Gold.
# ACMG/AMP treats Pathogenic and Likely pathogenic as both clinically actionable.
# Defined once here so every Gold table shares one authoritative definition.
PATHOGENIC_CATEGORIES = ["Pathogenic", "Likely pathogenic"]

print(f"\nTarget genome assembly : {GENOME_ASSEMBLY}")
print(f"Significance categories: {len(SIGNIFICANCE_CATEGORIES)} defined")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# ENVIRONMENT VALIDATION
# ─────────────────────────────────────────────────────────────────────────────
# Check Volume is reachable
# Check Spark SQL is working
# Check schema exists

print("=" * 60)
print("SETUP VALIDATION")
print("=" * 60)

all_ok = True

# Volume
try:
    dbutils.fs.ls(VOLUME_ROOT)
    print(f"  ✓ Volume reachable : {VOLUME_ROOT}")
except Exception as e:
    print(f"  ✗ Volume error     : {e}")
    all_ok = False

# Spark SQL
try:
    spark.sql("SELECT 1").collect()
    print(f"  ✓ Spark SQL        : working")
except Exception as e:
    print(f"  ✗ Spark SQL error  : {e}")
    all_ok = False

# Schema
try:
    spark.sql("USE " + CATALOG + "." + SCHEMA_BRONZE)
    print(f"  ✓ Schema           : {CATALOG}.{SCHEMA_BRONZE} reachable")
except Exception as e:
    print(f"  ✗ Schema error     : {e}")
    all_ok = False

print("=" * 60)
if all_ok:
    print("  SETUP COMPLETE — safe to run 01_bronze_ingestion")
else:
    print("  SETUP INCOMPLETE — fix errors above before proceeding")
print("=" * 60)